<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>RNN과 Seq2Seq(Recurrent Neural Network & Sequence-to-Sequence)</strong>에 대해 학습합니다.</br></br>
>순환 신경망의 구조와 인코더-디코더 아키텍처를 이해하고 학습해봅시다.

</br>

# RNN (Recurrent Neural Network)
> RNN은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">시퀀스 데이터의 순서 정보를 기억</mark>하며 처리하는 신경망입니다.
> 이전 시점의 은닉 상태(Hidden State)를 현재 입력과 함께 사용합니다.

일반 신경망(MLP)은 고정 크기의 입력 벡터를 받아 고정 크기의 출력을 내므로, 입력 간의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">순서(시간적 의존성)</mark>를 고려하지 못합니다. "나는 밥을 먹었다"와 "밥을 나는 먹었다"는 단어 집합은 같지만 의미가 다른데, MLP는 이 차이를 구분할 수 없어 언어, 음성, 시계열 데이터에 부적합합니다.</br>

RNN의 핵심 아이디어는 은닉 상태(Hidden State)로 과거를 기억하는 것입니다. RNN은 각 시점에서 현재 입력과 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이전 은닉 상태</mark>를 함께 받아 새로운 은닉 상태를 계산하며, 이 재귀적 구조 덕분에 시퀀스 앞부분의 정보가 뒷부분 처리에 영향을 미칩니다. 임베딩(Embedding)을 통해 정수 인덱스를 밀집 벡터로 변환한 뒤, 역전파(Backpropagation)로 가중치를 조정하며 학습합니다.</br>

나아가 번역("나는 학생이다" → "I am a student")처럼 입출력 길이가 다른 문제에서는 단순 RNN을 직접 적용할 수 없습니다. Seq2Seq는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인코더가 전체 입력을 하나의 컨텍스트 벡터로 압축</mark>하고, 디코더가 그 벡터로부터 출력을 순차 생성하는 방식으로 이 문제를 해결하며, 번역, 요약, 챗봇 등 현대 NLP 핵심 태스크의 기초 아키텍처입니다.

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$

</br>

## nn.Embedding
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어를 고정 크기의 밀집 벡터로 변환</mark>하는 룩업 테이블입니다.

In [ ]:
# TODO 1: 필요한 모듈을 불러오고, 임베딩 레이어를 생성한 뒤, 정수 텐서 [1, 5, 3]을 임베딩하여 shape를 출력해봅시다.

import torch
import torch.nn as nn

embedding = nn.Embedding(num_embeddings=100, embedding_dim=256)

x = torch.LongTensor([1, 5, 3])
embedded = embedding(x)
print(f"입력 shape: {x.shape}")
print(f"임베딩 shape: {embedded.shape}")
print(f"임베딩 벡터 (앞 5개): {embedded[0, :5].data.round(decimals=3)}")

</br>

## nn.RNN

In [ ]:
# TODO 2: RNN 레이어를 생성하고, 임베딩 결과에 배치 차원을 추가하여 입력한 뒤 출력과 은닉 상태의 shape를 출력해봅시다.

rnn = nn.RNN(input_size=256, hidden_size=512, num_layers=1, batch_first=True)

# (batch=1, seq_len=3, embed_dim=256)
input_seq = embedded.unsqueeze(0)
output, hidden = rnn(input_seq)

print(f"output shape: {output.shape}")    # 모든 시점의 출력
print(f"hidden shape: {hidden.shape}")    # 마지막 은닉 상태

💡RNN의 한계
> 긴 시퀀스에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기 소실(Vanishing Gradient)</mark> 문제가 발생합니다.</br>
> 이를 해결하기 위해 LSTM, GRU가 개발되었습니다.

</br>

# Seq2Seq (Sequence-to-Sequence)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인코더가 입력을 압축하고, 디코더가 출력을 생성</mark>하는 아키텍처입니다.

</br>

## Encoder 구현

In [ ]:
# TODO 3: Encoder 클래스를 정의하여 임베딩과 RNN을 초기화하고, 순전파에서 컨텍스트 벡터를 반환한 뒤, 인스턴스를 생성하여 테스트해봅시다.

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return hidden   # 컨텍스트 벡터

encoder = Encoder(vocab_size=100, embed_dim=256, hidden_dim=512)
source = torch.LongTensor([[1, 5, 3, 7]])   # (batch=1, seq_len=4)
context = encoder(source)
print(f"입력: {source.shape} → Context Vector: {context.shape}")

</br>

## Decoder 구현

In [ ]:
# TODO 4: Decoder 클래스를 정의하여 임베딩, RNN, 출력 선형 레이어를 초기화하고, 순전파에서 예측과 은닉 상태를 반환한 뒤, 시작 토큰과 컨텍스트를 입력하여 예측 shape를 출력해봅시다.

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output)
        return prediction, hidden

decoder = Decoder(vocab_size=100, embed_dim=256, hidden_dim=512)
target = torch.LongTensor([[2]])   # 시작 토큰
pred, new_hidden = decoder(target, context)
print(f"예측 shape: {pred.shape}")   # (batch, 1, vocab_size)
print(f"예측 토큰: {pred.argmax(-1).item()}")

💡왜 Seq2Seq가 필요한가?
> 입력과 출력의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">길이가 다른 경우</mark>(번역, 요약)에 고정 길이 모델로는 처리할 수 없습니다.</br>
> Seq2Seq는 가변 길이 입력을 가변 길이 출력으로 변환합니다.